**⚠️ Technical Note:** This notebook was developed in a Google Colab environment. The raw bioclimatic and GEDI/VHM datasets are hosted in a private Google Drive directory. To replicate this study, users must provide their own raster assets or contact the author for access to the standardized 1km² Parquet files.

# 01: Bioclimatic Grid Standardization
**Project:** A Validated Predictive Framework for Reforestation in Armenia

**Author:** Narek Ohanyan

In [ ]:
!pip install -q rioxarray geopandas shap rasterio zarr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.3 MB/s eta 0:00:00


In [ ]:
import os
import geopandas as gpd
import rioxarray
import warnings
import gc
import requests
from tqdm import tqdm
import xarray as xr
import numpy as np
from rasterio.enums import Resampling

In [ ]:
# Base Directory
BASE_DIR = '/content/drive/MyDrive/FORACCA_Output_1_2'

In [ ]:

warnings.filterwarnings("ignore")
os.environ['GTIFF_SRS_SOURCE'] = 'EPSG'

# Paths
SHAPEFILE_PATH = f'{BASE_DIR}/data_raw/arm_admin0.geojson'
OUT_DIR = f'{BASE_DIR}/data_raw/CHELSA_Clipped'
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Load Armenia Boundary and get Bounding Box (minx, miny, maxx, maxy)
armenia_gdf = gpd.read_file(SHAPEFILE_PATH)
armenia_geom = armenia_gdf.geometry
bounds = armenia_gdf.total_bounds

# 2. Parameters
variables = ['pr', 'tasmax', 'vpd']
years = range(1979, 2023)
months = [f"{m:02d}" for m in range(1, 13)]

print("Starting Sequential RAM-Safe Pipeline...")

for var in variables:
    for year in years:
        for month in months:
            out_filename = f"{OUT_DIR}/Armenia_{var}_{month}_{year}.tif"

            if os.path.exists(out_filename):
                continue

            url = f"https://os.unil.cloud.switch.ch/chelsa02/chelsa/global/monthly/{var}/{year}/CHELSA_{var}_{month}_{year}_V.2.1.tif"

            try:
                # Open with minimal overhead
                with rioxarray.open_rasterio(url, chunks=True) as rds:
                    # SLICE FIRST
                    # It crops the global file to a box before doing the heavy mask.
                    clipped_rds = rds.rio.clip_box(*bounds)

                    # Exact mask to Armenia's borders
                    final_rds = clipped_rds.rio.clip(armenia_geom, rds.rio.crs)

                    # Save
                    final_rds.rio.to_raster(out_filename)

                print(f"Done: {var} {year}-{month}")

                # Immediate cleanup
                del final_rds, clipped_rds
                gc.collect()

            except Exception as e:
                print(f"Error at {var} {year}-{month}: {e}")

Starting Sequential RAM-Safe Pipeline...
Done: pr 1984-01
Done: pr 1984-02
Done: pr 1984-03
Done: pr 1984-04
Done: pr 1984-05
Done: pr 1984-06
Done: pr 1984-07
Done: pr 1984-08
Done: pr 1984-09
Done: pr 1984-10
Done: pr 1984-11
Done: pr 1984-12
Done: pr 1985-01
Done: pr 1985-02
Done: pr 1985-03
Done: pr 1985-04
Done: pr 1985-05
Done: pr 1985-06
Done: pr 1985-07
Done: pr 1985-08
Done: pr 1985-09
Done: pr 1985-10
Done: pr 1985-11
Done: pr 1985-12
Done: pr 1986-01
Done: pr 1986-02
Done: pr 1986-03
Done: pr 1986-04
Done: pr 1986-05
Done: pr 1986-06
Done: pr 1986-07
Done: pr 1986-08
Done: pr 1986-09
Done: pr 1986-10
Done: pr 1986-11
Done: pr 1986-12
Done: pr 1987-01
Done: pr 1987-02
Done: pr 1987-03
Done: pr 1987-04
Done: pr 1987-05
Done: pr 1987-06
Done: pr 1987-07
Done: pr 1987-08
Done: pr 1987-09
Done: pr 1987-10
Done: pr 1987-11
Done: pr 1987-12
Done: pr 1988-01
Done: pr 1988-02
Done: pr 1988-03
Done: pr 1988-04
Done: pr 1988-05
Done: pr 1988-06
Done: pr 1988-07
Done: pr 1988-08
Done: p

In [ ]:


# 1. Define Paths
DATA_RAW_DIR = f'{BASE_DIR}/data_raw'
URL = "https://www.envidat.ch/dataset/d5dd3a3d-093e-4206-9e0b-76c6f21c9be1/resource/511abee2-19eb-45e9-b28c-23a3f8cada46/download/s2-vhm_2017_max.tif"
DESTINATION = os.path.join(DATA_RAW_DIR, "s2-vhm_2017_max.tif")

os.makedirs(DATA_RAW_DIR, exist_ok=True)

def download_large_file(url, destination):
    if os.path.exists(destination):
        print(f"File already exists at: {destination}")
        return

    print(f"Starting download: {url}")


    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    block_size = 1024 * 1024  # 1 Megabyte chunks

    with open(destination, 'wb') as file, tqdm(
        desc="Downloading VHM",
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(block_size):
            size = file.write(data)
            bar.update(size)

    print(f"\n✅ Successfully saved to: {destination}")

# Execute
download_large_file(URL, DESTINATION)

Starting download: https://www.envidat.ch/dataset/d5dd3a3d-093e-4206-9e0b-76c6f21c9be1/resource/511abee2-19eb-45e9-b28c-23a3f8cada46/download/s2-vhm_2017_max.tif



✅ Successfully saved to: /content/drive/MyDrive/FORACCA_Output_1_2/data_raw/s2-vhm_2017_max.tif


In [ ]:
# 1. Define Paths
VHM_30M_PATH = f'{BASE_DIR}/data_raw/s2-vhm_2017_max.tif'
CHELSA_TEMPLATE_PATH = f'{BASE_DIR}/data_raw/CHELSA_Clipped/Armenia_tasmax_02_1979.tif' # We use one file just to copy the 1km grid
OUT_VHM_NC = f'{BASE_DIR}/data_processed/Armenia_VHM_1km.nc'

print("Loading 30m VHM and 1km CHELSA Template...")

# 2. Load the data
# We open the template to steal its exact 1km geospatial grid
template_1km = rioxarray.open_rasterio(CHELSA_TEMPLATE_PATH).squeeze()
# We open the 30m VHM (and drop extra bands if they exist)
vhm_30m = rioxarray.open_rasterio(VHM_30M_PATH).squeeze()

# Replace any negative no-data values with 0 (bare ground)
vhm_30m = vhm_30m.where(vhm_30m >= 0, 0)

print("Aggregating to 1km... Calculating Mean...")
# 3. Calculate Mean(X)
vhm_mean_1km = vhm_30m.rio.reproject_match(
    template_1km,
    resampling=Resampling.average
)

print("Calculating Standard Deviation σ(H)...")
# 4. Calculate Mean(X^2)
vhm_squared = vhm_30m ** 2
vhm_sq_mean_1km = vhm_squared.rio.reproject_match(
    template_1km,
    resampling=Resampling.average
)

# 5. Apply the Variance Formula: Var = Mean(X^2) - Mean(X)^2
vhm_variance_1km = vhm_sq_mean_1km - (vhm_mean_1km ** 2)
# Ensure no tiny negative numbers from floating point math, then square root for Std Dev
vhm_variance_1km = vhm_variance_1km.where(vhm_variance_1km > 0, 0)
vhm_std_1km = np.sqrt(vhm_variance_1km)

print("Packaging and saving Master VHM NetCDF...")
# 6. Combine into a clean Xarray Dataset
vhm_dataset = xr.Dataset({
    'vhm_mean': vhm_mean_1km,
    'vhm_std': vhm_std_1km  # This is your crucial σ(H) for the FVS formula!
})

# Add spatial coordinate metadata back to the dataset
vhm_dataset = vhm_dataset.rio.write_crs(template_1km.rio.crs)

# Save to the data_processed folder
vhm_dataset.to_netcdf(OUT_VHM_NC)

print(f"✅ Success! Master VHM saved to: {OUT_VHM_NC}")
print("Notebook 01 is completely finished.")

Loading 30m VHM and 1km CHELSA Template...
Aggregating to 1km... Calculating Mean...
Calculating Standard Deviation σ(H)...
Packaging and saving Master VHM NetCDF...
✅ Success! Master VHM saved to: /content/drive/MyDrive/FORACCA_Output_1_2/data_processed/Armenia_VHM_1km.nc
Notebook 01 is completely finished.
